In [1]:
import pandas as pd
import numpy as np

from sklearn import datasets
from sklearn import ensemble

from evidently.legacy.pipeline.column_mapping import ColumnMapping
from evidently import Report
from evidently.legacy.metric_preset import ClassificationPreset, RegressionPreset
from evidently.metrics import *

In [2]:
#Dataset for regression
housing_data = datasets.fetch_california_housing(as_frame=True) #auto 
housing = housing_data.frame

housing.rename(columns={'MedHouseVal': 'target'}, inplace=True)
housing['prediction'] = housing_data['target'].values + np.random.normal(0, 3, housing.shape[0])

housing_ref = housing.sample(n=5000, replace=False)
housing_cur = housing.sample(n=5000, replace=False)

In [ ]:
housing_cur.head()

In [4]:
from evidently import DataDefinition, Regression

data_definition=DataDefinition(
        regression=[Regression(target="target", prediction="prediction")]
    )

In [5]:
from evidently import Dataset

reference_dataset = Dataset.from_pandas(
    pd.DataFrame(housing_ref),
    data_definition=data_definition,

)

In [6]:
current_dataset = Dataset.from_pandas(
    pd.DataFrame(housing_cur),
    data_definition=data_definition,
)

In [ ]:
regression_report = Report([
    MeanError(),
    MAE(),
    MAPE(),
    RMSE(),
    R2Score(),
    AbsMaxError(),
    DummyMAE(),
    DummyMAPE(),
    DummyRMSE(),
])

regression_snapshot = regression_report.run(current_dataset, reference_dataset)
regression_snapshot